# Notebook 08 — Multi-Physics Pipeline

## What you will learn
1. How to chain multiple simulations into a single automated pipeline
2. How data flows between stages (structural → EM → thermal)
3. How to run a parametric sweep over multiple configurations
4. How to checkpoint and resume a failed pipeline
5. The origami EMI pipeline as a worked example

---

## Part 1: The Pipeline Concept

### Why chain simulations?

Many engineering problems require sequential multi-physics:

```
1. Structural deformation  →  deformed geometry
2. EM simulation on deformed geometry  →  S-parameters
3. Power absorbed = A × P_incident  →  heat source distribution
4. Thermal simulation  →  temperature field
5. Thermal expansion  →  modified stress state
6. Back to structural...  (coupled loop)
```

The `SimulationPipeline` class makes this explicit and automated.

### Data flow

Each stage receives:
- Its own config (merged from global config + stage overrides)
- Named outputs from prior stages (via `input_map`)

Each stage returns:
- A flat dict of named outputs (file paths, numpy arrays, scalars)

These outputs are stored in the `context` dict and available to all later stages.

In [ ]:
import sys; sys.path.insert(0, '..')

# ── Show the pipeline API ──────────────────────────────────────────────────
from ams.multiphysics.pipeline import SimulationPipeline, PipelineStage, _deep_merge

# Simple example: two dummy stages
def stage_geometry(cfg, **kwargs):
    print(f'  Geometry stage: problem={cfg["problem"]["name"]}')
    return {'cdb_path': 'outputs/mesh.cdb', 'n_elements': 5000}

def stage_structural(cfg, cdb_path=None, **kwargs):
    print(f'  Structural stage: loading from {cdb_path}')
    return {'stl_path': 'outputs/deformed.stl', 'max_disp_m': 0.0045}

def stage_em(cfg, stl_path=None, **kwargs):
    print(f'  EM stage: using geometry from {stl_path}')
    import numpy as np
    freq = np.linspace(1, 20, 50)
    SE   = 15 + 10 * np.log10(freq)  # synthetic SE curve
    return {'SE_dB': SE, 'freq_GHz': freq}

# Build pipeline
pipeline = SimulationPipeline(
    stages=[
        PipelineStage('geometry',   run_fn=stage_geometry,   config={}),
        PipelineStage('structural', run_fn=stage_structural, config={},
                       input_map={'geometry.cdb_path': 'cdb_path'}),
        PipelineStage('em',         run_fn=stage_em,         config={},
                       input_map={'structural.stl_path': 'stl_path'}),
    ],
    global_cfg={'problem': {'name': 'demo_pipeline', 'physics': 'structural'}},
    output_dir='../outputs/pipeline_demo',
)

print('Pipeline plan:')
pipeline.print_plan()

print('\nRunning pipeline...')
results = pipeline.run()

print('\nPipeline outputs:')
for stage_name, outputs in results.items():
    print(f'  [{stage_name}]: {list(outputs.keys())}')

In [ ]:
# ── Visualize the EM stage output ─────────────────────────────────────────
import matplotlib.pyplot as plt

em_results = results['em']
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(em_results['freq_GHz'], em_results['SE_dB'], 'b-', linewidth=2)
ax.set_xlabel('Frequency (GHz)'); ax.set_ylabel('SE (dB)')
ax.set_title('Shielding Effectiveness — Pipeline Output')
ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

print(f'Mean SE: {em_results["SE_dB"].mean():.1f} dB')
print(f'Peak SE: {em_results["SE_dB"].max():.1f} dB @ {em_results["freq_GHz"][em_results["SE_dB"].argmax()]:.1f} GHz')

In [ ]:
# ── Parametric sweep ──────────────────────────────────────────────────────
import numpy as np

# Sweep over different fold angles (represented as scaling the displacement)
def stage_structural_sweep(cfg, cdb_path=None, **kwargs):
    fold_angle = cfg.get('fold_angle_deg', 30)
    scale = fold_angle / 30.0
    return {'stl_path': f'outputs/deformed_{fold_angle}deg.stl', 'max_disp_m': 0.005 * scale}

def stage_em_sweep(cfg, stl_path=None, **kwargs):
    fold_angle = cfg.get('fold_angle_deg', 30)
    freq = np.linspace(1, 20, 50)
    # SE increases with fold angle (more material in the path)
    SE = 15 + 10 * np.log10(freq) + fold_angle * 0.2
    return {'SE_dB': SE, 'freq_GHz': freq, 'fold_angle_deg': fold_angle}

sweep_pipeline = SimulationPipeline(
    stages=[
        PipelineStage('geometry',   run_fn=stage_geometry,         config={}),
        PipelineStage('structural', run_fn=stage_structural_sweep, config={},
                       input_map={'geometry.cdb_path': 'cdb_path'}),
        PipelineStage('em',         run_fn=stage_em_sweep,         config={},
                       input_map={'structural.stl_path': 'stl_path'}),
    ],
    global_cfg={'problem': {'name': 'sweep', 'physics': 'structural'}},
    output_dir='../outputs/sweep_demo',
)

fold_angles = [10, 20, 30, 45, 60, 75, 90]
param_sets  = [{'fold_angle_deg': a} for a in fold_angles]

print(f'Running {len(param_sets)}-point parametric sweep...')
all_results = sweep_pipeline.sweep(param_sets)
print(f'Done: {len(all_results)} runs complete')

In [ ]:
# ── Plot sweep results: SE vs fold angle ──────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.cm as cm

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

colors = cm.viridis(np.linspace(0, 1, len(all_results)))

for i, (res, color) in enumerate(zip(all_results, colors)):
    em = res['em']
    angle = fold_angles[i]
    ax1.plot(em['freq_GHz'], em['SE_dB'], color=color, label=f'{angle}°')

ax1.set_xlabel('Frequency (GHz)'); ax1.set_ylabel('SE (dB)')
ax1.set_title('SE vs Frequency — Fold Angle Sweep')
ax1.legend(fontsize=7, ncol=2); ax1.grid(True, alpha=0.3)

# Mean SE vs fold angle
mean_se = [res['em']['SE_dB'].mean() for res in all_results]
ax2.plot(fold_angles, mean_se, 'b-o', markersize=8, linewidth=2)
ax2.set_xlabel('Fold angle (degrees)'); ax2.set_ylabel('Mean SE (dB)')
ax2.set_title('Mean SE vs Fold Angle')
ax2.grid(True, alpha=0.3)

plt.tight_layout(); plt.show()

print('\nSweep summary:')
print(f'  {'Fold angle':>12}  {"Mean SE":>10}  {"Peak SE":>10}')
for angle, res in zip(fold_angles, all_results):
    SE = res['em']['SE_dB']
    print(f'  {angle:>12}°  {SE.mean():>10.1f} dB  {SE.max():>10.1f} dB')

---

## Part 2: Checkpoint and Resume

For long pipelines (each stage takes hours), checkpointing prevents re-running
completed stages after a crash:

```python
pipeline = SimulationPipeline(
    stages=[
        PipelineStage('structural', run_fn=..., checkpoint_key='structural'),
        PipelineStage('em',         run_fn=..., checkpoint_key='em'),
    ],
    output_dir='outputs/run_001',
)
results = pipeline.run()
# → creates outputs/run_001/checkpoint_structural.json
#              outputs/run_001/checkpoint_em.json

# If the EM stage crashes, restart:
# The structural checkpoint is loaded automatically.
# Only the EM stage re-runs.
results = pipeline.run()  # structural skipped (checkpoint found)
```

## Summary

```python
from ams.multiphysics.pipeline import (
    SimulationPipeline, PipelineStage,
    make_mapdl_structural_stage, make_hfss_em_stage
)

# Standard origami pipeline
pipeline = SimulationPipeline(
    stages=[
        make_mapdl_structural_stage(),
        make_hfss_em_stage(),
    ],
    global_cfg=load_config('config.yaml'),
)

# Single run
results = pipeline.run()

# Parametric sweep
all_results = pipeline.sweep([{'fold_angle_deg': a} for a in [30, 45, 60, 90]])
```

**Next:** `09_custom_material_models.ipynb`